# SubSight Phase 1 gate v3: chrono split + cross-chunk test

Runs the honest-split comparison for pipe segmentation. Free Colab T4, 256px, seed 42. Runtime > GPU > T4 before running.

Expected output: one chrono-trained checkpoint plus IoU/Dice rows for chrono val, chrono test, and Chunk1-4 full-chunk. Paste the printed table back to fill the README gate.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!test -d SubSight || git clone https://github.com/manocw/SubSight.git 2>&1 | tail -2
%cd /content/SubSight
!git pull --ff-only 2>&1 | tail -1
!pip install -r requirements.txt 2>&1 | tail -2
import torch
print('cuda:', torch.cuda.is_available())

In [ ]:
# Download SubPipe archives from Zenodo (CC-BY-4.0, verify on page). Mini first, Mini2 only if chunks are missing.
# Resumable: rerunning this cell picks up where it stopped. Progress prints at most ~20s apart.
import os, socket, time, urllib.request, zipfile
from pathlib import Path

socket.setdefaulttimeout(60)

# Exact file links from the Zenodo API record (newer version id, /content pattern).
URLS = {
    'SubPipeMini.zip': 'https://zenodo.org/api/records/12666132/files/SubPipeMini.zip/content',
    'SubPipeMini2.zip': 'https://zenodo.org/api/records/12666132/files/SubPipeMini2.zip/content',
}
EXPECTED = {'SubPipeMini.zip': 6078303615, 'SubPipeMini2.zip': 4945761374}
Path('data').mkdir(exist_ok=True)

def fetch(name, retries=8):
    out = Path('data') / name
    total = EXPECTED[name]
    if out.exists() and out.stat().st_size >= 0.95 * total:
        print(f'{name}: already complete ({out.stat().st_size / 1e9:.2f} GB)')
        return out
    for attempt in range(1, retries + 1):
        have = out.stat().st_size if out.exists() else 0
        req = urllib.request.Request(URLS[name], headers={
            'Range': f'bytes={have}-', 'User-Agent': 'SubSight-colab'})
        try:
            r = urllib.request.urlopen(req, timeout=60)
            mode = 'ab'
            if have and r.status == 200:
                print('server ignored resume, restarting file')
                mode, have = 'wb', 0
            else:
                print(f'{name}: resuming at {have / 1e9:.2f} GB (attempt {attempt})')
            with open(out, mode) as f:
                last, mark = time.time(), have
                while True:
                    chunk = r.read(4 * 1024 * 1024)
                    if not chunk:
                        break
                    f.write(chunk)
                    f.flush()
                    if time.time() - last > 20:
                        done = f.tell()
                        rate = (done - mark) / (time.time() - last) / 1e6
                        print(f'  {done / 1e9:.2f}/{total / 1e9:.2f} GB '
                              f'({100 * done / total:.0f}%) {rate:.1f} MB/s', flush=True)
                        last, mark = time.time(), done
        except Exception as e:
            print(f'  stall or error: {e}, retrying in 10s...')
            time.sleep(10)
            continue
        size = out.stat().st_size
        if size >= 0.95 * total:
            print(f'{name}: done ({size / 1e9:.2f} GB)')
            return out
        print(f'  incomplete ({size / 1e9:.2f} GB), retrying...')
    raise RuntimeError(f'{name} failed after retries, rerun cell to continue')

def have_chunk(c):
    return Path(f'data/Chunk{c}/Segmentation').is_dir()

def extract_once(name):
    marker = Path(f'data/.extracted_{name}.ok')
    if marker.exists():
        print(f'{name}: already extracted')
        return
    print(f'extracting {name}...')
    with zipfile.ZipFile(Path('data') / name) as f:
        print('zip contents sample:', f.namelist()[:5])
        f.extractall('data')
    marker.touch()

z = fetch('SubPipeMini.zip')
extract_once('SubPipeMini.zip')

print('chunks present:', [c for c in range(5) if have_chunk(c)])
if not all(have_chunk(c) for c in range(5)):
    z2 = fetch('SubPipeMini2.zip')
    extract_once('SubPipeMini2.zip')
    print('after Mini2:', [c for c in range(5) if have_chunk(c)])

In [ ]:
# Sanity: pair counts per chunk + mask check on Chunk0 (uses repo code, read-only).
import sys
sys.path.insert(0, '.')
from src.dataset import find_pairs

for c in range(5):
    root = f'data/Chunk{c}/Segmentation'
    try:
        pairs = find_pairs(root)
        print(f'Chunk{c}: {len(pairs)} pairs')
    except FileNotFoundError:
        print(f'Chunk{c}: MISSING')

!python notebooks/inspect_masks.py --root data/Chunk0/Segmentation --n 3

In [ ]:
# Chrono retrain on Chunk0: timestamp order 60/20/20, test split locked, seed 42.
!sed -i 's/split: "random"/split: "chrono"/' configs/config.yaml
!grep -A2 'train_split' configs/config.yaml
!python -m src.train --config configs/config.yaml

In [ ]:
# Gate table: chrono val first, then every available chunk full (no retrain, no leakage).
import glob, subprocess

subprocess.run(['python', '-m', 'src.evaluate', '--checkpoint', 'checkpoints/best.pth', '--num-images', '6'], check=True)
for root in sorted(glob.glob('data/Chunk*/Segmentation')):
    print(f'\n=== {root} ===')
    subprocess.run(['python', '-m', 'src.evaluate', '--checkpoint', 'checkpoints/best.pth',
                    '--data-root', root, '--full-chunk', '--num-images', '2'], check=True)

In [ ]:
# Copy these numbers into the README Phase 1 gate table. Worst-frame note included.
!ls -la checkpoints/best.pth outputs/eval_examples.png
from google.colab import files
files.download('outputs/eval_examples.png')